In [1]:
#%pip install pdfplumber

import pdfplumber
import pandas as pd
import ast


In [2]:
def extract_text_from_pdf(pdf_path):
    page_data = []
    with pdfplumber.open(pdf_path) as pdf:
        all_pages_text = ""
        for page_number, page in enumerate(pdf.pages):
        
            all_text = page.extract_text() or ""

            tables = page.find_tables()

            # text_data=""
            # excluded_data=""

            for table in tables:
                bbox = table.bbox
                text_within_table = page.within_bbox(bbox).extract_text() or ""

                table_content = table.extract()
                df = pd.DataFrame(table_content)
                table_data = df.to_string(index=False, header=False)

                all_text = all_text.replace(text_within_table, table_data)
            
            # all_pages_text += all_text.strip()    
            # all_pages_text += "\n"
            page_data.append({
                "page_number": page_number+1,
                "page_text": all_text.strip()
            })
        
        df = pd.DataFrame(page_data)
        return df

pdf_path = r"./HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf"
df = extract_text_from_pdf(pdf_path)
print(df)

    page_number                                          page_text
0             1  Part A\n<<Date>>\n<<Master Policyholder’s Name...
1             2  MASTER POLICY DOCUMENT- HDFC Life Group Poorna...
2             3  POLICY SCHEDULE\n1. Master Policy Number:<< sy...
3             4  Part B\nDefinitions\nThe following capitalized...
4             5  i. Acute condition - Acute condition is a dise...
5             6  (38) Sum Assured- means the amount payable und...
6             7  Part C\n1. Benefits:\n(1) Benefits on Death or...
7             8  21. Progressive\nScleroderma 22. Muscular\nDys...
8             9  In case if the Scheme Member surrenders the Ce...
9            10  Part D\n1. Additional Sum Assured option:\nThe...
10           11  2. Assignment or Transfer\nAssignment shall be...
11           12  thereof, within 15 days from the date of recei...
12           13  Part E\n1. Additional Servicing Charges\nNil\n...
13           14  Part F\n1. Waiting Period and Exclusions:\ni.

In [3]:
print(df["page_text"][2])   


POLICY SCHEDULE
1. Master Policy Number:<< system/operations generated>>
2. Date of Proposal:<<<< system/operations generated>>
3. Date of Inception: <<Date of Inception>>
4. Effective Date:<< system/operations service generated>>
5. Master Policyholder:<<Name of Company/Group>>
6. Name of the Scheme:<<NAME of Scheme>>
7. Scheme Type:<<Compulsory/Voluntary>>
8. Plan option:<<Name of Plan option>>
9. Eligibility to join the Scheme for the Scheme Member:
         Eligibility                          Age (Last Birthday) (in years)                                                       NaN   NaN
Minimum Age at Entry                                                  < 18 >                                                       NaN   NaN
Maximum Age at Entry                                  Single Premium Payment                                               Life Option <79 >
                 NaN                                                     NaN Extra Life Option &\nAccelerated Critical I

In [4]:
# Please note : Since ChromaDB automatically embeds the data given embedding Function, we don't need to embed the data here.
# So this below code is not required. So it is commented.

#import openai

# 

# df2 = df.copy()

# def create_embeddings(df):
#     df["embeddings"] = None
#     for page_number, row in df.iterrows():
#         embeddings = openai.embeddings.create(input=[row["page_text"]], model="text-embedding-3-small")
#         df.at[page_number, "embeddings"] = embeddings.data[0].embedding
#     return df

# df2 = create_embeddings(df2)
# print(df2)



In [5]:
df['metadata'] = None
for index, row in df.iterrows():
    df.at[index, "metadata"] = f"Page Number: {row['page_number']} | Section: {row['page_text'][:100]}"

df.head(10)




,page_number,page_text,metadata
0,1,Part A\n<<Date>>\n<<Master Policyholder’s Name...,Page Number: 1 | Section: Part A\n<<Date>>\n<<...
1,2,MASTER POLICY DOCUMENT- HDFC Life Group Poorna...,Page Number: 2 | Section: MASTER POLICY DOCUME...
2,3,POLICY SCHEDULE\n1. Master Policy Number:<< sy...,Page Number: 3 | Section: POLICY SCHEDULE\n1. ...
3,4,Part B\nDefinitions\nThe following capitalized...,Page Number: 4 | Section: Part B\nDefinitions\...
4,5,i. Acute condition - Acute condition is a dise...,Page Number: 5 | Section: i. Acute condition -...
5,6,(38) Sum Assured- means the amount payable und...,Page Number: 6 | Section: (38) Sum Assured- me...
6,7,Part C\n1. Benefits:\n(1) Benefits on Death or...,Page Number: 7 | Section: Part C\n1. Benefits:...
7,8,21. Progressive\nScleroderma 22. Muscular\nDys...,Page Number: 8 | Section: 21. Progressive\nScl...
8,9,In case if the Scheme Member surrenders the Ce...,Page Number: 9 | Section: In case if the Schem...
9,10,Part D\n1. Additional Sum Assured option:\nThe...,Page Number: 10 | Section: Part D\n1. Addition...


In [6]:
import chromadb
from chromadb.api.types import EmbeddingFunction
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# -------------------------------
# Local Embedding Function
# -------------------------------
class LocalEmbeddingFunction(EmbeddingFunction):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def __call__(self, input):
        if isinstance(input, str):
            input = [input]
        return self.model.encode(input).tolist()


# -------------------------------
# Chroma Client
# -------------------------------
client = chromadb.Client(
    Settings(
        anonymized_telemetry=False
    )
)

embedding_function = LocalEmbeddingFunction()

# -------------------------------
# ALWAYS create fresh collection
# -------------------------------
collections = [c.name for c in client.list_collections()]

if "insurance_policy_documents" in collections:
    main_collection = client.get_collection(
        "insurance_policy_documents",
        embedding_function=embedding_function
    )
else:
    main_collection = client.create_collection(
        name="insurance_policy_documents",
        embedding_function=embedding_function
    )


# -------------------------------
# Chunking Function
# -------------------------------
def chunk_text(text, max_words=300):
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

# -------------------------------
# Add Documents
# -------------------------------
if main_collection.count() == 0:
    for idx, row in df.iterrows():
        try:
            chunks = chunk_text(row["page_text"])
            metadata = {
                "page": row["page_number"],
                "preview": row["page_text"][:100]
            }

            for i, chunk in enumerate(chunks):
                main_collection.add(
                    documents=[chunk],
                    metadatas=[metadata],
                    ids=[f"{idx}_{i}"]
                )

            print(f"Added page {idx}")

        except Exception as e:
            print(f"Error adding page {idx}: {e}")
else:
    print("Documents already exist. Skipping ingestion.")




C:\Insurance_Document_Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Insurance_Document_Chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ak697\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order t

Added page 0
Added page 1
Added page 2
Added page 3
Added page 4
Added page 5
Added page 6
Added page 7
Added page 8
Added page 9
Added page 10
Added page 11
Added page 12
Added page 13
Added page 14
Added page 15
Added page 16
Added page 17
Added page 18
Added page 19
Added page 20
Added page 21
Added page 22
Added page 23
Added page 24
Added page 25
Added page 26
Added page 27
Added page 28
Added page 29
Added page 30


In [7]:
collection_list = client.list_collections()

def add_to_cache(docs, metadatas):
    """
    Stores retrieved answers into cache collection
    """
    for i, doc in enumerate(docs):
        try:
            cache_collection.add(
                documents=[doc],
                metadatas=[metadatas[i]],
                ids=[f"cache_{hash(doc)}"]
            )
        except:
            pass  # Avoid duplicate ID crash


if "insurance_policy_cache" not in [collection.name for collection in collection_list]:
    cache_collection = client.create_collection("insurance_policy_cache", embedding_function=embedding_function)
else:
    cache_collection = client.get_collection("insurance_policy_cache")

def query_cache_layer(query):
    #ids_to_check = [query + str(i) for i in range(3)]
    #print(ids_to_check)
    #print(cache_collection.count())
    results = cache_collection.query(
        query_texts=[query],
        n_results=1,
        #where={"ids": {"$in": ["what is Waiting Period and Exclusions?0"]}}
    )
    return results
#query_cache_layer("what is Waiting Period and Exclusions?")

In [8]:
#query = "What is the procedure to claim the insurance?"
#query_embedding = openai.embeddings.create(input=[query], model="text-embedding-3-small").data[0].embedding
def main_collection_query(query):
    results = main_collection.query(
        query_texts=[query],
        n_results=5
    )
    return results["documents"][0], results["metadatas"][0]

In [ ]:
from google import genai
import getpass

print("Enter Gemini API key:")
api_key = getpass.getpass()

client = genai.Client(api_key=api_key)


def chat_comm(user_message):
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=user_message
    )
    return response.text




Enter Gemini API key:


In [ ]:
def display_result(query, docs, metadatas):
    #print(docs)
    system_prompt = f"""
    You are a helpful assistant that can answer questions about the insurance policy document.
    You are given a question and a list of documents that are relevant to the question.
    You need to answer the question based on the documents.
    You need to display the answer in a readable format. It should be clear, concise and easy to understand.
    The Query is: {query}
    The Relevant Document/ documents are: {docs}
    Also, after the answer is given, give the source of the data so the user knows which document the answer is coming from.
    Use the Metadata as the reference of the data.
    Give all available references in the answer and ask them to check the documents for more information.
    The Metadata is: {metadatas}
    1 Shot example:
    **Reference 1: Page Number: 16 | Section: a. Completed claim form, (including NEFT details and bank account proof as specified in the claim**
    **Reference 2: Page Number: 12 | Section: thereof, within 15 days from the date of receipt of the Certificate of Insurance, as per IRDAI**
    etc
    List all available references in the answer at the end of the answer.
    If the answer is not found in the documents, say that you are not able to find the answer in the documents and tell them to contact the insurance provider or the agent/customer care.
    """
    #print(system_prompt)

    response = chat_comm(system_prompt)
    return response
    #"""Once the answer is given, give the source of the data so the user knows which document the answer is coming from.
    #The format of the answer should be:
      #"Source: {result["metadatas"][0]}""""

In [ ]:


from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_results(query, docs, metadatas):
    pairs = [(query, doc) for doc in docs]
    scores = reranker.predict(pairs)
    sorted_idx = scores.argsort()[::-1]
    return (
        [docs[i] for i in sorted_idx],
        [metadatas[i] for i in sorted_idx]
    )



In [ ]:
print("Welcome to the Insurance Policy Document RAG Chatbot.")
print("-" * 80)

user_query = input("Ask your question: ")

# 1️⃣ Check cache
cache_results = query_cache_layer(user_query)

if (
    cache_results
    and "documents" in cache_results
    and cache_results["documents"]
    and len(cache_results["documents"][0]) > 0
):
    print("Answering from cache...")
    docs = cache_results["documents"][0]
    metadatas = cache_results["metadatas"][0]
else:
    print("Searching main document...")
    docs, metadatas = main_collection_query(user_query)
    add_to_cache(docs, metadatas)


# 2️⃣ Rerank
docs, metadatas = rerank_results(user_query, docs, metadatas)

# 3️⃣ Ask Gemini
answer = display_result(user_query, docs[:3], metadatas[:3])
print("\nAnswer:\n")
print(answer)

print("\nThanks for using the chatbot!")

